# Proyecto Práctico HPC: ejecución secuencial vs paralela

**Integrantes:**
- Angel Dolores Galván de la Cruz (@galvandelacruzangeldolores)
- Pamela (@pamela-mtz)
- Rosa (@rosa1208)

## Objetivo
Hacer un programa en Python que procese muchos datos y comparar la ejecución **secuencial** contra la **paralela**, midiendo el tiempo con 1, 2 y 4 workers (3 corridas cada uno) para ver si paralelizar de verdad mejora el rendimiento.

In [ ]:
import sys, os, platform
sys.path.append(os.path.join("..", "src"))   # para poder importar los .py de src/

import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from hpc_core import version_secuencial, version_paralela, medir
from graficas import grafica_tiempo, grafica_speedup, grafica_eficiencia

## Hardware utilizado
Esta celda saca los datos de la compu donde se corrió.

In [ ]:
print("Sistema operativo:", platform.system(), platform.release())
print("Versión de Python:", platform.python_version())
print("Núcleos lógicos (os.cpu_count):", os.cpu_count())
print("Procesador:", platform.processor() or "(ver README)")

## 1. Desarrollo del programa
La función que se evalúa es:

$$f(x) = \sqrt{x} + x^2 + \sin(x) + \cos(x) + \log(x)$$

Se aplica a todos los valores de `x = 1 ... N` y se suman los resultados. El código está en `src/hpc_core.py`:
- **Secuencial:** un solo ciclo que recorre todos los datos.
- **Paralela:** se parte el rango en bloques y `multiprocessing.Pool` se los reparte a los workers; al final se suman los resultados parciales.

In [ ]:
N = 5_000_000          # cantidad de datos
WORKERS = [1, 2, 4]    # configuraciones a probar
REPETICIONES = 3       # veces que se corre cada una

Antes de medir, checamos que la versión paralela dé **el mismo resultado** que la secuencial (con un N chico para que sea rápido).

In [ ]:
import math
n_prueba = 200_000
r_seq = version_secuencial(n_prueba)
for w in WORKERS:
    r_par = version_paralela(n_prueba, w)
    print(f"{w} worker(s) -> mismo resultado que el secuencial: {math.isclose(r_seq, r_par, rel_tol=1e-9)}")

## 2. Experimentación
Se corre cada configuración 3 veces y se guarda el tiempo de cada una.

In [ ]:
tiempos_seq = [medir(version_secuencial, N)[0] for _ in range(REPETICIONES)]
tiempos = {w: [medir(version_paralela, N, w)[0] for _ in range(REPETICIONES)] for w in WORKERS}

print("Secuencial:", [round(t, 3) for t in tiempos_seq])
for w in WORKERS:
    print(f"{w} worker(s):", [round(t, 3) for t in tiempos[w]])

### Tabla de resultados
- Speedup: $S_p = T_1 / T_p$
- Eficiencia: $E_p = S_p / p$

donde $T_1$ es el tiempo promedio con 1 worker y $T_p$ el tiempo promedio con $p$ workers.

In [ ]:
prom = {w: sum(tiempos[w]) / len(tiempos[w]) for w in WORKERS}
T1 = prom[1]
speedup = {w: T1 / prom[w] for w in WORKERS}
eficiencia = {w: speedup[w] / w for w in WORKERS}

tabla = pd.DataFrame({
    "Workers": WORKERS,
    "Prueba 1 (s)": [tiempos[w][0] for w in WORKERS],
    "Prueba 2 (s)": [tiempos[w][1] for w in WORKERS],
    "Prueba 3 (s)": [tiempos[w][2] for w in WORKERS],
    "Promedio (s)": [prom[w] for w in WORKERS],
    "Speedup": [speedup[w] for w in WORKERS],
    "Eficiencia": [eficiencia[w] for w in WORKERS],
}).round(4)

print(f"Tiempo promedio secuencial: {sum(tiempos_seq)/len(tiempos_seq):.4f} s")
tabla

In [ ]:
os.makedirs(os.path.join("..", "resultados"), exist_ok=True)
tabla.to_csv(os.path.join("..", "resultados", "resumen.csv"), index=False)

## 3. Gráficas de rendimiento
Se generan con código (están en `src/graficas.py`) y se guardan en la carpeta `resultados/`.

In [ ]:
grafica_tiempo(WORKERS, [prom[w] for w in WORKERS])
grafica_speedup(WORKERS, [speedup[w] for w in WORKERS])
grafica_eficiencia(WORKERS, [eficiencia[w] for w in WORKERS])

## 4. Análisis de resultados
Las respuestas completas están en el `README.md` del repositorio.

## 5. Git, GitHub y GitFlow
El proyecto se hizo con las ramas `main`, `develop` y `feature/...`, con el flujo `feature/*` → Pull Request → `develop` → Pull Request → `main`. La explicación completa está en el `README.md`.